# S1 — Ingest and clip

Stage 1 of the Manhattan Sidewalk Shade Index pipeline.

Downloads street trees, sidewalk polygons, LION centerlines, and the borough boundary using the endpoints resolved in `s0_resolve_sources.ipynb` (`data/SOURCES.md`), filters, clips to Manhattan, reprojects to EPSG:32618, and writes GeoParquet to `data/interim/` for S2.

Server-side filtering (SoQL `$where`) is used at download time where possible — trees by `borocode`/`status`, sidewalks by a Manhattan bounding box — so this stage doesn't pull citywide data just to throw most of it away; a precise polygon clip to the real Manhattan boundary still happens locally afterward.

**Accept when:** Manhattan tree count is plausible (order 10⁴–10⁵ per CLAUDE.md §5), all layers share EPSG:32618, bounds overlap and cover Manhattan, no null geometries.

In [1]:
import re
import zipfile
from pathlib import Path
from datetime import datetime

import requests
import yaml
import pandas as pd
import geopandas as gpd
import fiona

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
UA = {"User-Agent": "Mozilla/5.0"}

RAW = PROJECT_ROOT / config["raw_dir"]
INTERIM = PROJECT_ROOT / config["interim_dir"]
RAW.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

ANALYSIS_CRS = config["analysis_crs"]
print("s1: Ingest and clip to Manhattan")
print(f"Timestamp: {datetime.now().isoformat()}\n")

Cannot find header.dxf (GDAL_DATA is not defined)


s1: Ingest and clip to Manhattan
Timestamp: 2026-08-28T13:37:43.468093



## Parse `data/SOURCES.md` for resolved dataset IDs

Re-derives the endpoint each source resolved to in S0, rather than hardcoding IDs here — if S0 is re-run against a catalog refresh, S1 picks up the new IDs automatically.

In [2]:
def parse_sources_md(path: Path) -> dict[str, dict]:
    text = path.read_text()
    sections = re.split(r"\n### ", text)[1:]
    parsed = {}
    for section in sections:
        key, _, body = section.partition("\n")
        key = key.strip()
        entry = {}
        for field in ["Status", "Dataset ID", "Asset kind", "URL"]:
            m = re.search(rf"\*\*{field}:\*\*\s*(.+)", body)
            if m:
                entry[field.lower().replace(" ", "_")] = m.group(1).strip()
        parsed[key] = entry
    return parsed


sources = parse_sources_md(PROJECT_ROOT / "data" / "SOURCES.md")
for key in ["street_trees_primary", "sidewalks", "street_centerlines", "borough_boundary"]:
    assert sources.get(key, {}).get("status") == "success", f"{key} not resolved successfully in SOURCES.md"
print("Parsed resolved sources:", list(sources.keys()))

Parsed resolved sources: ['street_trees_primary', 'street_trees_alt', 'borough_boundary', 'sidewalks', 'street_centerlines']


## Download raw data

Trees: filter server-side to Manhattan + Alive. Sidewalks: filter server-side to a Manhattan bounding box (SKILL.md §8's NYC bbox). Borough boundary: all 5 boroughs (tiny). LION: ships as an Esri File Geodatabase inside the zip (not a shapefile, despite CLAUDE.md's expectation) — download, extract, and read the `lion` layer (129 columns; `Street` for naming, `LBoro`/`RBoro` for side-of-street borough, native CRS EPSG:2263, NY State Plane feet).

In [3]:
trees_path = RAW / "trees_manhattan.json"
if not trees_path.exists():
    # NOTE: this dataset's .geojson export returns geometry=null (lat/lon are plain
    # number columns, not a Socrata Point type) -- pull .json and build points from
    # latitude/longitude ourselves instead.
    trees_id = sources["street_trees_primary"]["dataset_id"]
    url = f"https://data.cityofnewyork.us/resource/{trees_id}.json"
    params = {"borocode": "1", "status": "Alive", "$limit": 100_000}
    resp = requests.get(url, params=params, headers=UA, timeout=120)
    resp.raise_for_status()
    trees_path.write_bytes(resp.content)
print(f"trees: {trees_path.stat().st_size:,} bytes")

trees: 57,606,439 bytes


In [4]:
sidewalks_path = RAW / "sidewalks_manhattan_bbox.geojson"
if not sidewalks_path.exists():
    sw_id = sources["sidewalks"]["dataset_id"]
    url = f"https://data.cityofnewyork.us/resource/{sw_id}.geojson"
    # Manhattan bbox per modern-gis/SKILL.md §8: xmin>-74.02, xmax<-73.93, ymin>40.70, ymax<40.82
    where = "within_box(the_geom, 40.82, -74.02, 40.70, -73.93)"
    params = {"$where": where, "$limit": 50_000}
    resp = requests.get(url, params=params, headers=UA, timeout=120)
    resp.raise_for_status()
    sidewalks_path.write_bytes(resp.content)
print(f"sidewalks: {sidewalks_path.stat().st_size:,} bytes")

sidewalks: 58,051,409 bytes


In [5]:
borough_path = RAW / "borough_boundaries.geojson"
if not borough_path.exists():
    boro_id = sources["borough_boundary"]["dataset_id"]
    url = f"https://data.cityofnewyork.us/resource/{boro_id}.geojson"
    resp = requests.get(url, params={"$limit": 10}, headers=UA, timeout=60)
    resp.raise_for_status()
    borough_path.write_bytes(resp.content)
print(f"borough boundaries: {borough_path.stat().st_size:,} bytes")

borough boundaries: 3,167,672 bytes


In [6]:
lion_zip_path = RAW / "lion.zip"
lion_extract_dir = RAW / "lion"
if not lion_zip_path.exists():
    lion_url = sources["street_centerlines"]["url"]
    with requests.get(lion_url, headers=UA, timeout=300, stream=True) as resp:
        resp.raise_for_status()
        with open(lion_zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                f.write(chunk)
print(f"LION zip: {lion_zip_path.stat().st_size:,} bytes")

if not lion_extract_dir.exists():
    lion_extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(lion_zip_path) as zf:
        zf.extractall(lion_extract_dir)

lion_gdb_candidates = list(lion_extract_dir.rglob("*.gdb"))
assert lion_gdb_candidates, "No .gdb found in extracted LION zip"
lion_gdb_path = lion_gdb_candidates[0]
print("LION geodatabase:", lion_gdb_path)
print("layers:", fiona.listlayers(lion_gdb_path))

LION zip: 45,981,902 bytes
LION geodatabase: C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\raw\lion\lion\lion.gdb
layers: ['node', 'node_stname', 'altnames', 'lion']


## Load, filter, reproject, clip

In [7]:
trees_df = pd.read_json(trees_path)
trees_raw = gpd.GeoDataFrame(
    trees_df,
    geometry=gpd.points_from_xy(
        pd.to_numeric(trees_df["longitude"]), pd.to_numeric(trees_df["latitude"])
    ),
    crs="EPSG:4326",
)
print(f"trees_raw: {len(trees_raw):,} rows, CRS={trees_raw.crs}")

trees_raw["tree_dbh"] = pd.to_numeric(trees_raw["tree_dbh"], errors="coerce")
trees = trees_raw[
    (trees_raw["status"] == config["tree_status_filter"])
    & (trees_raw["tree_dbh"] > config["tree_min_dbh_cm"])
    & trees_raw.geometry.notna()
].copy()
assert config["tree_dbh_input_unit"] == "inches"
trees["tree_dbh_cm"] = trees["tree_dbh"] * 2.54
print(f"trees after filter: {len(trees):,} rows")

trees_raw: 62,427 rows, CRS=EPSG:4326
trees after filter: 62,416 rows


In [8]:
sidewalks_raw = gpd.read_file(sidewalks_path)
print(f"sidewalks_raw: {len(sidewalks_raw):,} rows, CRS={sidewalks_raw.crs}")

borough = gpd.read_file(borough_path)
borough["borocode"] = pd.to_numeric(borough["borocode"], errors="coerce")
manhattan_boundary_4326 = borough[borough["borocode"] == config["manhattan_borocode"]].dissolve()
assert len(manhattan_boundary_4326) == 1, "Expected exactly one Manhattan boundary feature"
print(f"Manhattan boundary bounds (4326): {manhattan_boundary_4326.total_bounds}")

C:\Users\juanz\miniconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


sidewalks_raw: 6,121 rows, CRS=EPSG:4326
Manhattan boundary bounds (4326): [-74.04772963  40.68291695 -73.906651    40.87903805]


In [9]:
lion_layer = gpd.read_file(lion_gdb_path, layer="lion")
print(f"Using LION segment layer: {len(lion_layer):,} rows, CRS={lion_layer.crs}")

street_name_col = "Street"
assert street_name_col in lion_layer.columns
# LBoro/RBoro carry the borough on each side of the centerline -- useful for S2's
# side-of-street assignment later.
assert {"LBoro", "RBoro"}.issubset(lion_layer.columns)
print(f"street name column: {street_name_col}")

Using LION segment layer: 243,237 rows, CRS=EPSG:2263
street name column: Street


In [10]:
# Reproject everything to the analysis CRS, then clip to Manhattan precisely
manhattan_boundary = manhattan_boundary_4326.to_crs(ANALYSIS_CRS)

trees_m = trees.to_crs(ANALYSIS_CRS)
trees_clipped = gpd.clip(trees_m, manhattan_boundary)

sidewalks_m = sidewalks_raw.to_crs(ANALYSIS_CRS)
sidewalks_clipped = gpd.clip(sidewalks_m, manhattan_boundary)

lion_m = lion_layer.to_crs(ANALYSIS_CRS)
lion_clipped = gpd.clip(lion_m, manhattan_boundary)

print(f"trees clipped: {len(trees_clipped):,}")
print(f"sidewalks clipped: {len(sidewalks_clipped):,}")
print(f"lion clipped: {len(lion_clipped):,}")

trees clipped: 62,416
sidewalks clipped: 4,584
lion clipped: 34,174


## QA summary

In [11]:
def qa_summary(label, gdf):
    print(f"  {label}:")
    print(f"    Rows: {len(gdf):,}")
    print(f"    CRS: {gdf.crs}")
    print(f"    Bounds: {gdf.total_bounds}")
    print(f"    Null geometries: {gdf.geometry.isna().sum()}")


print("\n" + "=" * 70)
print("S1 — Ingest and clip: QA Summary")
print("=" * 70)
qa_summary("trees", trees_clipped)
qa_summary("sidewalks", sidewalks_clipped)
qa_summary("lion", lion_clipped)
print("=" * 70)

assert str(trees_clipped.crs) == ANALYSIS_CRS, "trees CRS mismatch"
assert str(sidewalks_clipped.crs) == ANALYSIS_CRS, "sidewalks CRS mismatch"
assert str(lion_clipped.crs) == ANALYSIS_CRS, "lion CRS mismatch"
assert 1e4 <= len(trees_clipped) <= 1e5, f"tree count {len(trees_clipped)} outside plausible 10^4-10^5 range"
assert trees_clipped.geometry.isna().sum() == 0
assert sidewalks_clipped.geometry.isna().sum() == 0
assert lion_clipped.geometry.isna().sum() == 0
print("All S1 acceptance checks passed.")


S1 — Ingest and clip: QA Summary
  trees:
    Rows: 62,416
    CRS: EPSG:32618
    Bounds: [ 582920.42354543 4506122.30600164  591747.34522004 4525227.99433398]
    Null geometries: 0
  sidewalks:
    Rows: 4,584
    CRS: EPSG:32618
    Bounds: [ 582852.00406689 4506040.35491024  590272.11036483 4519303.7455418 ]
    Null geometries: 0
  lion:
    Rows: 34,174
    CRS: EPSG:32618
    Bounds: [ 580462.11603013 4504026.77712393  592124.36939455 4525900.69178863]
    Null geometries: 0
All S1 acceptance checks passed.


## Write GeoParquet

In [12]:
trees_clipped.to_parquet(PROJECT_ROOT / config["output"]["ingested_trees"])
sidewalks_clipped.to_parquet(PROJECT_ROOT / config["output"]["ingested_sidewalks"])
lion_clipped.to_parquet(PROJECT_ROOT / config["output"]["ingested_lion"])
print("Wrote:")
print(f"  {config['output']['ingested_trees']}")
print(f"  {config['output']['ingested_sidewalks']}")
print(f"  {config['output']['ingested_lion']}")
print("\ns1 complete. Ready for S2 (analysis units).")

Wrote:
  data/interim/trees.parquet
  data/interim/sidewalks.parquet
  data/interim/lion.parquet

s1 complete. Ready for S2 (analysis units).
